In [ ]:
library(dplyr)
library(tidyr)
library(pheatmap)
library(viridis)
library(tools)
library(cluster)
library(igraph)
library(tibble)

library(clusterProfiler)
library(org.Hs.eg.db)
library(dplyr)

In [1]:
# 读入
signature_Neu <- read.csv(
  "Example_refbuilder_Neutrophils/3.ALL_cGEP_top_genes_Neutrophils_wide_filt.csv",
  check.names = FALSE,
  stringsAsFactors = FALSE
)

# 转成 list（核心）
cluster_genes <- lapply(signature_Neu, function(x) {
  x[!is.na(x) & x != ""]
})
cluster_genes

$cGEP1
 [1] "CD3G"      "IL32"      "LCK"       "CCL5"      "CD3E"      "GZMA"     
 [7] "NKG7"      "CD2"       "TRBC2"     "ITM2A"     "CD3D"      "CD27"     
[13] "TRAC"      "IKZF3"     "GZMH"      "RASGRP1"   "TRAT1"     "CTLA4"    
[19] "ETS1"      "GZMK"      "CD8B"      "CD8A"      "SYNE2"     "TRBC1"    
[25] "CTSW"      "CST7"      "LINC00426" "CD7"       "PRF1"      "SKAP1"    

$cGEP2
 [1] "CXCR5"     "LINC01641" "GAS5"      "RPS8"      "RPS18"     "RPS2"     
 [7] "RPL37A"    "RPLP2"     "RPL23A"    "RPS23"     "RPS12"     "RPL18A"   
[13] "RPL10A"    "RPL3"      "RPS27"     "RPS29"     "RPL13"     "RPL13A"   
[19] "EEF1A1"    "RPS6"      "RPL41"     "RPL10"     "TRBC1"     "RPS3"     
[25] "RPL4"      "RPS17"     "SNHG6"     "RPS20"     "RPS14"     "RPL19"    

$cGEP3
 [1] "S100A9"   "S100A12"  "S100A8"   "VCAN"     "LYZ"      "PLBD1"   
 [7] "VIM"      "SERPINB1" "METTL9"   "HP"       "TKT"      "MNDA"    
[13] "PADI4"    "GCA"      "GAPDH"    "APLP2"    "SLC2A3"   "ITGAM"   
[19] "IRS2"     "IVNS1ABP" "CKAP4"    "HMGB2"    "MXD1"     "CES1"    
[25] "TSPO"     "GM2A"     "TALDO1"   "ANXA1"    "SELL"     "RBP7"    

$cGEP4
 [1] "IER3"       "IL1B"       "NFKBIA"     "NLRP3"      "AC095055.1"
 [6] "TNFAIP3"    "VEGFA"      "KDM6B"      "ZNF331"     "INSIG1"    
[11] "GPR183"     "MAP3K8"     "AC092368.3" "EREG"       "CXCL2"     
[16] "GRASP"      "CXCL8"      "AC016831.7" "RASGEF1B"   "PLAUR"     
[21] "NR4A1"      "THBS1"      "NAMPT"      "CD83"       "ARF4-AS1"  
[26] "CLHC1"      "HLA-DQB2"   "PIM3"       "MRPS24"     "CSRNP1"    

$cGEP5
 [1] "IFIT1"      "RSAD2"      "IFIT3"      "ISG15"      "MX1"       
 [6] "HERC5"      "IFIT2"      "GBP1"       "CMPK2"      "SPDEF"     
[11] "AC010501.1" "OAS2"       "AC138907.8" "IFI44L"     "SERPING1"  
[16] "IFITM3"     "GBP5"       "GBP4"       "IFI6"       "AC007362.1"
[21] "EPSTI1"     "RNF213"     "DDX58"      "AC148477.3" "UBE2L6"    
[26] "XAF1"       "STAT1"      "AC113189.2" "LINC02453"  "TNFSF13B"  

$cGEP6
 [1] "FCGR3A"     "RYR3"       "AIF1"       "C1QA"       "LST1"      
 [6] "IFITM3"     "RHOC"       "SMIM25"     "CDKN1C"     "C1QB"      
[11] "FCER1G"     "CLIC2"      "CD79B"      "ABI3"       "CSF1R"     
[16] "AC007038.1" "TCF7L2"     "ATP8B1"     "FMNL2"      "CEBPB"     
[21] "TRNP1"      "LINC02449"  "DRAP1"      "AC124045.1" "COTL1"     
[26] "MTSS1"      "LRRC25"     "L1TD1"      "RPS19"      "C1QC"      

$cGEP7
 [1] "HLA-DQA1"   "FCER1A"     "HLA-DRA"    "HLA-DPA1"   "HLA-DQA2"  
 [6] "RGS1"       "HLA-DPB1"   "HLA-DQB1"   "HLA-DRB1"   "HLA-DRB5"  
[11] "CLEC10A"    "CD74"       "HIST1H3A"   "HLA-DMA"    "CD1C"      
[16] "CCSER1"     "AC102953.2" "AL390728.5" "ZNF697"     "MRC1"      
[21] "ZNF790"     "HLA-DOA"    "TNFRSF21"   "LINC01970"  "AC124068.2"
[26] "CST7"       "SH3BP4"     "ZNF597"     "TNFRSF12A"  "RGL1"

In [2]:
convert_gene <- function(gene_symbol_vector){
  bitr(gene_symbol_vector, 
       fromType = "SYMBOL",
       toType = "ENTREZID",
       OrgDb = org.Hs.eg.db) %>%
    pull(ENTREZID)
}

clean_symbol <- function(x){
  x <- unique(x)
  x <- x[x != ""]
  x <- toupper(x)
  x
}

cluster_genes <- lapply(cluster_genes, clean_symbol)


In [3]:
library(clusterProfiler)
library(org.Hs.eg.db)
library(ReactomePA)
library(msigdbr)
library(dplyr)

# ============================
# 清洗基因名
# ============================
clean_symbol <- function(x){
  x <- unique(toupper(x))
  x[x != ""]
}

cluster_genes <- lapply(cluster_genes, clean_symbol)

# ============================
# SYMBOL → ENTREZ 映射
# ============================
all_symbols <- unique(unlist(cluster_genes))

symbol2entrez <- bitr(all_symbols,
                      fromType="SYMBOL",
                      toType="ENTREZID",
                      OrgDb=org.Hs.eg.db)

# named vector
map_table <- symbol2entrez$ENTREZID
names(map_table) <- symbol2entrez$SYMBOL

# ============================
# MSigDB Hallmark gene sets
# ============================
msig_h <- msigdbr(species = "Homo sapiens", category = "H")

hallmark_list <- msig_h %>% 
  split(.$entrez_gene) %>%
  lapply(function(x) x$gs_name)

# ============================
# 初始化结果表
# ============================
kegg_list <- list()
go_bp_list <- list()
reactome_list <- list()
hallmark_list_results <- list()

# ============================
# 循环 77 clusters
# ============================
for(cl in names(cluster_genes)){

  message("Processing ", cl, " ...")

  # SYMBOL → ENTREZ
  genes <- cluster_genes[[cl]]
  genes_entrez <- map_table[genes]
  genes_entrez <- genes_entrez[!is.na(genes_entrez)]

  if(length(genes_entrez) < 5){
    warning(cl, ": <5 valid genes → skip")
    next
  }

  # KEGG（可以为空）
  kegg_list[[cl]] <- tryCatch({
    enrichKEGG(gene = genes_entrez, organism = "hsa",
               pvalueCutoff = 0.1)
  }, error=function(e) NULL)

  # GO BP
  go_bp_list[[cl]] <- enrichGO(gene = genes_entrez, 
                               OrgDb = org.Hs.eg.db,
                               keyType="ENTREZID",
                               ont="BP",
                               pvalueCutoff=0.1)

  # Reactome（强烈推荐）
  reactome_list[[cl]] <- enrichPathway(gene = genes_entrez,
                                       pvalueCutoff = 0.1,
                                       readable = TRUE)

  # Hallmark enrichment
  hallmark_list_results[[cl]] <- enricher(
    gene = genes_entrez,
    TERM2GENE = msig_h[,c("gs_name","entrez_gene")],
    pvalueCutoff = 0.1
  )
}

Warning message:
“程辑包‘clusterProfiler’是用R版本4.2.2 来建造的”


Registered S3 methods overwritten by 'treeio':
  method              from    
  MRCA.phylo          tidytree
  MRCA.treedata       tidytree
  Nnode.treedata      tidytree
  Ntip.treedata       tidytree
  ancestor.phylo      tidytree
  ancestor.treedata   tidytree
  child.phylo         tidytree
  child.treedata      tidytree
  full_join.phylo     tidytree
  full_join.treedata  tidytree
  groupClade.phylo    tidytree
  groupClade.treedata tidytree
  groupOTU.phylo      tidytree
  groupOTU.treedata   tidytree
  is.rooted.treedata  tidytree
  nodeid.phylo        tidytree
  nodeid.treedata     tidytree
  nodelab.phylo       tidytree
  nodelab.treedata    tidytree
  offspring.phylo     tidytree
  offspring.treedata  tidytree
  parent.phylo        tidytree
  parent.treedata     tidytree
  root.treedata       tidytree
  rootnode.phylo      tidytree
  sibling.phylo       tidytree

clusterProfiler v4.6.0  For help: https://yulab-smu.top/bi

In [4]:
hallmark_list_results

$cGEP1
#
# over-representation test
#
#...@organism 	 UNKNOWN 
#...@ontology 	 UNKNOWN 
#...@gene 	 chr [1:30] "917" "9235" "3932" "6352" "916" "3001" "4818" "914" "28638" ...
#...pvalues adjusted by 'BH' with cutoff <0.1 
#...2 enriched terms found
'data.frame':	2 obs. of  9 variables:
 $ ID         : chr  "HALLMARK_ALLOGRAFT_REJECTION" "HALLMARK_COMPLEMENT"
 $ Description: chr  "HALLMARK_ALLOGRAFT_REJECTION" "HALLMARK_COMPLEMENT"
 $ GeneRatio  : chr  "13/18" "5/18"
 $ BgRatio    : chr  "200/4383" "200/4383"
 $ pvalue     : num  1.78e-14 9.91e-04
 $ p.adjust   : num  2.49e-13 6.94e-03
 $ qvalue     : num  2.06e-13 5.74e-03
 $ geneID     : chr  "917/3932/6352/916/3001/914/915/50852/2113/926/925/924/5551" "3932/6352/3001/10125/3003"
 $ Count      : int  13 5
#...Citation
 T Wu, E Hu, S Xu, M Chen, P Guo, Z Dai, T Feng, L Zhou, W Tang, L Zhan, X Fu, S Liu, X Bo, and G Yu.
 clusterProfiler 4.0: A universal enrichment tool for interpreting omics data.
 The Innovation. 2021, 2(3):100141 




In [5]:
library(openxlsx)
library(dplyr)

wb <- createWorkbook()

# ===== 1. Hallmark =====
hallmark_df <- bind_rows(
  lapply(names(hallmark_list_results), function(cl){
    res <- hallmark_list_results[[cl]]@result
    if (is.null(res) || nrow(res)==0) return(NULL)
    res$Cluster <- cl
    res
  })
)

addWorksheet(wb, "Hallmark")
writeData(wb, "Hallmark", hallmark_df)

# ===== 2. Reactome =====
reactome_df <- bind_rows(
  lapply(names(reactome_list), function(cl){
    res <- reactome_list[[cl]]@result
    if (is.null(res) || nrow(res)==0) return(NULL)
    res$Cluster <- cl
    res
  })
)

addWorksheet(wb, "Reactome")
writeData(wb, "Reactome", reactome_df)

# ===== 3. GO BP =====
go_bp_df <- bind_rows(
  lapply(names(go_bp_list), function(cl){
    res <- go_bp_list[[cl]]@result
    if (is.null(res) || nrow(res)==0) return(NULL)
    res$Cluster <- cl
    res
  })
)

addWorksheet(wb, "GO_BP")
writeData(wb, "GO_BP", go_bp_df)

# ===== 保存 Excel 文件 =====
saveWorkbook(wb, "./3.2.Neu_pathway/3.2.1.Neu_GEP7_Cluster_Pathway_Results.xlsx", overwrite = TRUE)

In [9]:
library(openxlsx)
library(dplyr)
library(readr)
library(stringr)

# ==============================================================================
# 1. 读取 Jaccard 注释结果
# ==============================================================================
JACCARD_FILE <- "/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.5.cNMF_Neutrophils/3.Neutrophils_GEP_Anno/3.Neutrophil_cGEP7_Annotation_Results.csv"

if (file.exists(JACCARD_FILE)) {
  anno_df <- read_csv(JACCARD_FILE, show_col_types = FALSE)
  # 强制重命名第一列为 Cluster
  colnames(anno_df)[1] <- "Cluster"
  cat("✅ 成功读取 CSV。列名预览:", colnames(anno_df), "\n")
} else {
  stop("❌ 找不到文件！")
}

# ==============================================================================
# 2. 定义辅助函数 (已修复 select 冲突)
# ==============================================================================

# 函数1: 提取 Top 3 通路摘要
extract_top_summary <- function(enrich_list, db_prefix) {
  df_list <- lapply(names(enrich_list), function(cl) {
    res <- enrich_list[[cl]]@result
    
    if (is.null(res) || nrow(res) == 0) {
      return(data.frame(Cluster = cl, Pathways = "", stringsAsFactors = FALSE))
    }
    
    # 筛选显著并取前3
    top_res <- res %>% 
      filter(p.adjust < 0.05) %>% 
      arrange(p.adjust) %>% 
      head(3)
    
    path_str <- paste(top_res$Description, collapse = " | ")
    
    data.frame(Cluster = cl, Pathways = path_str, stringsAsFactors = FALSE)
  })
  
  combined_df <- bind_rows(df_list)
  
  # 重命名列 (使用 Base R 避免报错)
  target_col <- paste0("Top_", db_prefix)
  colnames(combined_df)[colnames(combined_df) == "Pathways"] <- target_col
  
  return(combined_df)
}

# 函数2: 提取详情 (【核心修复】：使用 dplyr::select)
extract_detail_df <- function(enrich_list) {
  bind_rows(lapply(names(enrich_list), function(cl) {
    res <- enrich_list[[cl]]@result
    if (is.null(res) || nrow(res) == 0) return(NULL)
    
    res$Cluster <- cl
    
    # 【这里修改了】：强制使用 dplyr::select，防止被 AnnotationDbi 覆盖
    res %>% dplyr::select(Cluster, everything())
  }))
}

# ==============================================================================
# 3. 执行提取与合并
# ==============================================================================
cat("正在提取 Hallmark...\n")
hallmark_sum <- extract_top_summary(hallmark_list_results, "Hallmark")
hallmark_det <- extract_detail_df(hallmark_list_results)

cat("正在提取 Reactome...\n")
reactome_sum <- extract_top_summary(reactome_list, "Reactome")
reactome_det <- extract_detail_df(reactome_list)

cat("正在提取 GO_BP...\n")
go_bp_sum    <- extract_top_summary(go_bp_list, "GO_BP")
go_bp_det    <- extract_detail_df(go_bp_list)

# 合并到总表
final_anno_df <- anno_df %>%
  left_join(hallmark_sum, by = "Cluster") %>%
  left_join(reactome_sum, by = "Cluster") %>%
  left_join(go_bp_sum,    by = "Cluster")

# ==============================================================================
# 4. 写入 Excel
# ==============================================================================
OUT_XLSX <- "./3.2.Neu_pathway/3.2.1.Neutrophils_GEP7_Cluster_Pathway_Master_Annotation.xlsx"
wb <- createWorkbook()

# Sheet 1: 总览
addWorksheet(wb, "Summary_Overview")
writeData(wb, "Summary_Overview", final_anno_df)
freezePane(wb, "Summary_Overview", firstRow = TRUE)

# Sheet 2: Hallmark 详情
addWorksheet(wb, "Detail_Hallmark")
writeData(wb, "Detail_Hallmark", hallmark_det)

# Sheet 3: Reactome 详情
addWorksheet(wb, "Detail_Reactome")
writeData(wb, "Detail_Reactome", reactome_det)

# Sheet 4: GO_BP 详情
addWorksheet(wb, "Detail_GO_BP")
writeData(wb, "Detail_GO_BP", go_bp_det)

saveWorkbook(wb, OUT_XLSX, overwrite = TRUE)

cat("--------------------------------------------------\n")
cat("✅ 成功！所有冲突已修复。\n")
cat("📄 文件已保存至:", OUT_XLSX, "\n")
cat("--------------------------------------------------\n")

✅ 成功读取 CSV。列名预览: Cluster Spectra_Label Overlap_Count Jaccard_Score Key_Markers Top_5_Genes_Input 
正在提取 Hallmark...
正在提取 Reactome...
正在提取 GO_BP...
--------------------------------------------------
✅ 成功！所有冲突已修复。
📄 文件已保存至: ./3.2.Neu_pathway/3.2.1.Neutrophils_GEP7_Cluster_Pathway_Master_Annotation.xlsx 
--------------------------------------------------
